In [25]:
import pandas as pd
import numpy as np
 

In [26]:
rating_data = pd.io.parsers.read_csv('ratings.dat',names=["user_id","movie_id",'rating','time'],delimiter="::",engine="python")
movie_data = pd.io.parsers.read_csv("movies.dat",engine="python",names=["movie_id","title",'genre'],delimiter="::",encoding = "ISO-8859-1")

In [33]:
print(rating_data)
print(movie_data)


         user_id  movie_id  rating       time
0              1      1193       5  978300760
1              1       661       3  978302109
2              1       914       3  978301968
3              1      3408       4  978300275
4              1      2355       5  978824291
...          ...       ...     ...        ...
1000204     6040      1091       1  956716541
1000205     6040      1094       5  956704887
1000206     6040       562       5  956704746
1000207     6040      1096       4  956715648
1000208     6040      1097       4  956715569

[1000209 rows x 4 columns]
      movie_id                               title  \
0            1                    Toy Story (1995)   
1            2                      Jumanji (1995)   
2            3             Grumpier Old Men (1995)   
3            4            Waiting to Exhale (1995)   
4            5  Father of the Bride Part II (1995)   
...        ...                                 ...   
3878      3948             Meet the Parent

In [34]:
ratings_matrix = np.ndarray(
    shape= (np.max(rating_data.movie_id.values),np.max(rating_data.user_id.values)),
    dtype= np.uint8
)

ratings_matrix[rating_data.movie_id.values-1,rating_data.user_id.values-1] = rating_data.rating.values
print(ratings_matrix)

[[5 0 0 ... 0 0 3]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [35]:
normalized_matrix = ratings_matrix - np.asarray([(np.mean(ratings_matrix,axis=1))]).T
print(normalized_matrix)

[[ 3.57400662 -1.42599338 -1.42599338 ... -1.42599338 -1.42599338
   1.57400662]
 [-0.37152318 -0.37152318 -0.37152318 ... -0.37152318 -0.37152318
  -0.37152318]
 [-0.23874172 -0.23874172 -0.23874172 ... -0.23874172 -0.23874172
  -0.23874172]
 ...
 [-0.03278146 -0.03278146 -0.03278146 ... -0.03278146 -0.03278146
  -0.03278146]
 [-0.02582781 -0.02582781 -0.02582781 ... -0.02582781 -0.02582781
  -0.02582781]
 [-0.24288079 -0.24288079 -0.24288079 ... -0.24288079 -0.24288079
  -0.24288079]]


In [36]:
A = normalized_matrix.T / np.sqrt(ratings_matrix.shape[0] - 1)
print(A)

U, S, V = np.linalg.svd(A)

[[ 0.05685934 -0.00591061 -0.00379817 ... -0.00052152 -0.0004109
  -0.00386402]
 [-0.02268632 -0.00591061 -0.00379817 ... -0.00052152 -0.0004109
  -0.00386402]
 [-0.02268632 -0.00591061 -0.00379817 ... -0.00052152 -0.0004109
  -0.00386402]
 ...
 [-0.02268632 -0.00591061 -0.00379817 ... -0.00052152 -0.0004109
  -0.00386402]
 [-0.02268632 -0.00591061 -0.00379817 ... -0.00052152 -0.0004109
  -0.00386402]
 [ 0.02504108 -0.00591061 -0.00379817 ... -0.00052152 -0.0004109
  -0.00386402]]


In [42]:
def cosine_similarity(ratingdata,movie_id,top_n):
    index = movie_id - 1
    movie_row = ratingdata[index,:]
    
    magnitude = np.sqrt(np.einsum('ij,ij -> i',ratingdata,ratingdata))
    cosine_sim = np.dot(movie_row,ratingdata.T) / (magnitude[index] * magnitude)
    sort_indices = np.argsort(-cosine_sim)
    
    return sort_indices[:top_n]
 


In [46]:
k = 50
movie_id = 15
top_n = 10

sliced = V.T[:,:k]
indexes = cosine_similarity(ratingdata=sliced,movie_id=movie_id,top_n=top_n)

print("Recomentation for Movie {0}:\n".format(
    movie_data[movie_data.movie_id == movie_id].title.values[0])
)

for m_id in indexes + 1:
    print(movie_data[movie_data.movie_id == m_id].title.values[0])
    


Recomentation for Movie Cutthroat Island (1995):

Cutthroat Island (1995)
Batman & Robin (1997)
Anaconda (1997)
Avengers, The (1998)
Batman Forever (1995)
Dante's Peak (1997)
Chain Reaction (1996)
Dragonheart (1996)
Allan Quartermain and the Lost City of Gold (1987)
Cobra (1925)


c:\Users\v.rajkumar\AppData\Local\Programs\Python\Python37\lib\site-packages\ipykernel_launcher.py:6: RuntimeWarning: invalid value encountered in true_divide
  
